In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# Attention으로 번역기 만들기
- Attention

## 1. 패키지 import 및 하이퍼파라미터
- 하이터 파라미터 : 모델의 정확도 및 학습속도에 영향을 미치는 변수

In [2]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout, LSTM, Attention, Concatenate
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128 # hidden layer units 설정
MY_EPOCH = 500

## 2. 번역 데이터 불러오기

In [3]:
raw = pd.read_csv('data/translate.csv', header=None) # 영어 알파벳 4개 -> 한글 번역 2글자로
eng_kor = raw.values.tolist() # DataFrame => list
print('영어-한글 번역 데이터 :',eng_kor[:3])
print('영어-한글 번역 데이터 수 :',len(eng_kor))

영어-한글 번역 데이터 : [['cold', '감기'], ['come', '오다'], ['cook', '요리']]
영어-한글 번역 데이터 수 : 110


## 3. 영어 알파벳과 한글 문자 리스트 만들기

In [4]:
# 영어 알파벳 리스트(e_alpha)
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
# print(e_alpha)
# {c:i for i, c in enumerate(e_alpha)}

# 한글 문자 리스트(k_ch,k_alpha)
k_ch = sorted(set(''.join([data[1] for data in eng_kor])))
# print(k_ch)
k_alpha = pd.read_csv('data/korean.csv', header=None)[0].tolist()
k_alpha == k_ch # 내용(값,index)이 모두 같을 때

# 순서는 무시하고 내용만 같은 지
from collections import Counter
list1 = ['가','간','나']
list2 = ['간','나','가']
list_1_count = Counter(list1)
list_2_count = Counter(list2)
list_1_count== list_2_count

True

In [5]:
alpha = e_alpha + k_ch
print('영어와 한글 알파벳 :',alpha)
alpha_total_size = len(alpha)
print('전체 알파벳 개수(원핫인코딩 할 size) :',alpha_total_size)
print('한글 알파벳 개수 :',len(k_alpha))

영어와 한글 알파벳 : ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
전체 알파벳 개수(원핫인코딩 할 size) : 171
한글 알파벳 개수 : 142


## 4. 문자당 num을 갖는 dict 만들기

In [6]:
char_to_num = {c:i for i,c in enumerate(alpha)}
num_to_char = {i:c for i,c in enumerate(alpha)}
print(char_to_num, num_to_char)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 29, '각': 30, '간': 31, '감': 32, '개': 33, '거': 34, '것': 35, '게': 36, '계': 37, '고': 38, '관': 39, '광': 40, '구': 41, '굴': 42, '규': 43, '그': 44, '금': 45, '기': 46, '깊': 47, '나': 48, '날': 49, '남': 50, '내': 51, '넓': 52, '녀': 53, '노': 54, '놀': 55, '농': 56, '높': 57, '뉴': 58, '늦': 59, '다': 60, '단': 61, '도': 62, '동': 63, '들': 64, '람': 65, '랑': 66, '래': 67, '램': 68, '류': 69, '름': 70, '릎': 71, '리': 72, '많': 73, '망': 74, '매': 75, '머': 76, '먼': 77, '멍': 78, '메': 79, '명': 80, '모': 81, '목': 82, '무': 83, '물': 84, '미': 85, '바': 86, '반': 87, '방': 88, '번': 89, '복': 90, '부': 91, '분': 92, '붕': 93, '비': 94, '뿌': 95, '사': 96, '상': 97, '색': 98, '생': 99, '서': 100, '선': 101, '소': 102, '손': 103, '수': 104, '쉽': 105, '스': 106, '시': 107, '식': 108, '실': 109, '싸': 110,

In [7]:
data = eng_kor[0]
print(data)
print('인코더 입력 :',char_to_num['c'],char_to_num['o'],char_to_num['l'],char_to_num['d'])
print('인코더 입력(원핫인코딩 전) :',[char_to_num[c] for c in data[0]])
print('디코더 입력(원핫인코딩 전) :',[char_to_num[c] for c in 'S' + data[1]])
print('디코더 출력(원핫인코딩 X) :',[char_to_num[c] for c in data[1]+'E']) # 원핫 인코딩 대신 loss = sparse_categorical_crossentropy

['cold', '감기']
인코더 입력 : 5 17 14 6
인코더 입력(원핫인코딩 전) : [5, 17, 14, 6]
디코더 입력(원핫인코딩 전) : [0, 32, 46]
디코더 출력(원핫인코딩 X) : [32, 46, 1]


In [55]:
# 희소행렬의 원핫 인코딩 방법 1 (희소행렬에서는 pd.get_dummies([2,8,7]) 사용 불가(size 조절이 불가 -> 전체 원소개수만 가능))

In [8]:
# pd.get_dummies([5,17,14,6]) # 사용 불가
to_categorical([5,7,6,8],
#                num_classes=alpha_total_size, # 실제로 사용했어야 하는 값
              num_classes=10, # 학생의 학습을 위해 교본의 사이즈를 10으로 고정해서 사용
              )

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]], dtype=float32)

In [9]:
# 희소행렬의 원핫 인코딩 방법 2
np.eye(10)[[5,7,6,8]]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]])

## 5. 인코더 입력, 디코더 입력, 디코더 출력
- 인코더 입력 데이터 : 영어 알파벳 -> 숫자 -> 원핫인코딩, shape = (110, 4, 171) array
- 디코더 입력 데이터 : 'S' + 한글 문자 -> 숫자 -> 원핫인코딩, shape = (110, 1('S') + 2 ,171) array
- 디코더 출력 데이터 : 한글 문자 + 'E' -> 숫자 shape = (110, 2+1('E')) list => 최종 shape (110, 3, 1) array로

In [11]:
def encoding(eng_kor = eng_kor) :
    ''' 
    [['eng4' : 'kor2']] 형태 데이터가 입력되면 인코더 입력, 디코더 입력, 디코더 출력 순으로 반환함
    '''
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력
    for data in eng_kor :
        # 인코더 입력 데이터(영어 알파벳 -> 숫자 -> 원핫인코딩)
        eng = [char_to_num[c] for c in data[0]]
        eng_one = np.eye(alpha_total_size)[eng]
#         print('영어 :',eng, eng_one)
        enc_in.append(eng_one) # eng_one의 shape = (4,171)
    
        # 디코더 입력 데이터('S' + 한글 문자 -> 숫자 -> 원핫인코딩)
        kor = [char_to_num[c] for c in 'S' + data[1]]
        kor_one = to_categorical(kor,num_classes=alpha_total_size) # np.eye(alpha_total_size)[kor]
#         print('한글 :',kor,kor_one)
        dec_in.append(kor_one) # kor_one의 shape = (1('S')+2 = 3 ,171)
        
        # 디코더 출력 데이터(한글 문자 + 'E' -> 숫자)
        kor = [char_to_num[c] for c in data[1] + 'E']
#         print(kor)
        dec_out.append(kor)
    return enc_in, dec_in, dec_out

In [12]:
sample = [['cold','감기'],['wood','나무']]
x_enc, x_dec, y_dec = encoding(sample)
X_enc = np.array(x_enc)
X_enc.shape # shape (2 단어수, 4 단어 당 문자 수, 171 학습에 필요한, 단어를 찢어서 만든 문자의 수)
X_dec = np.array(x_dec)
X_dec.shape # shape (2 단어수, 3 "S단어"의 문자 수, 171 학습에 필요한, 단어를 찢어서 만든 문자의 수)
Y_dec = np.array(y_dec)
Y_dec.shape # shape (2 단어 수, 3"단어E"의 문자수), 원핫인코딩 X loss = sparse_categorical_crossentropy로 해결, 그러나 입력하는 값들과 차원을 맞춰야함

(2, 3)

In [13]:
# shape (2 단어 수, 3"단어E"의 문자수), 원핫인코딩 X loss = sparse_categorical_crossentropy로 해결, 그러나 입력하는 값들과 차원을 맞춰야함
# 축 증가 방법 1
Y_dec.reshape(2,3,1)
# 축 증가 방법 2 : 맨 마지막 축 증가하는 함수
np.expand_dims(Y_dec, axis=-1)
# 축 증가 방법 3
Y_dec[...,np.newaxis]
# 축 증가 방법 4
Y_dec[:,:,None]

array([[[32],
        [46],
        [ 1]],

       [[48],
        [83],
        [ 1]]])

## 6. 전체 입력데이터, 타겟데이터 준비

In [14]:
x_inc, x_dec, y_dec = encoding()
type(x_inc),type(x_dec),type(y_dec) # 타입 변환 필요
X_enc = np.array(x_inc)
X_dec = np.array(x_dec)
# Y_dec = np.array(y_dec)[...,np.newaxis]
Y_dec = np.expand_dims(y_dec,-1) # 리스트 타입에서도 바로 사용이 가능
X_enc.shape, X_dec.shape, Y_dec.shape

((110, 4, 171), (110, 3, 171), (110, 3, 1))

## 7. 모델 구현

### 7.1 Seq2Seq

In [122]:
# 인코더 LSTM
ENC_IN = Input((X_enc.shape[1],alpha_total_size))
_, state_h, state_c = LSTM(units = MY_HIDDEN # 128
         ,return_state=True,
#          return_sequences=True, # LSTM 윗 출력 안받음
                          )(ENC_IN) 

# 인코더와 디코더 연결 고리
LINK = [state_h, state_c]

# 디코더 LSTM
DEC_IN = Input(shape = (Y_dec.shape[1],alpha_total_size))
DEC_MID = LSTM(units = MY_HIDDEN,
#               return_state=False, # 기본값
              return_sequences=True, # 윗출력을 받겠다 => 시퀀스 정보가 담긴 데이터를 받겠다
              )(DEC_IN,
               initial_state=LINK)

# 최종 출력층
DEC_OUT = Dense(units = alpha_total_size,
               activation='softmax')(DEC_MID)

# 모델
model = Model(inputs=[ENC_IN,DEC_IN],
              outputs = DEC_OUT)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_6 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_7 (InputLayer)           [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_5 (LSTM)                  [(None, 128),        153600      ['input_6[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

## 7.2 Attention

In [16]:
# 인코더 입력
ENC_IN = Input(shape=(4,alpha_total_size)) # alpha_total_size = 171

# 인코더 LSTM : 모든 방향성 자료 출력 필요. return_sequences=True / 결과값 출력, return_state=True / state_h, state_c 
ENC_OUt, state_h, state_c = LSTM(units = MY_HIDDEN,
                                return_sequences=True,
                                return_state=True)(ENC_IN)

# state_h, state_c 전송 계층
LINK = [state_h,state_c]

# 디코더 입력
DEC_IN = Input(shape=(3,alpha_total_size))

# 디코더 LSTM : 상위방향, 결과값 전송 필요(return_sequences = True)
DEC_LSTM_OUT, _, _ = LSTM(units=MY_HIDDEN, # 128
                          return_sequences=True,
                          return_state=True)(DEC_IN,
                                            initial_state=LINK)

# Attention
CONTEXT_VECTOR = Attention()([DEC_LSTM_OUT,ENC_OUt])

# 컨텍스트 벡터와 디코더 LSTM 결과를 Concatenate
CONTEXT_AND_LSTM_OUT= Concatenate()([CONTEXT_VECTOR,
                                     DEC_LSTM_OUT])
OUT = Dense(units = alpha_total_size, activation='softmax')(CONTEXT_AND_LSTM_OUT)

# 모델 정의
model = Model(inputs=[ENC_IN,DEC_IN],outputs=OUT)
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_4 (InputLayer)           [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_2 (LSTM)                  [(None, 4, 128),     153600      ['input_3[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                            

## 8. 모델 학습과정 설정 & 학습

In [17]:
model.compile(loss = 'sparse_categorical_crossentropy',
             optimizer = 'rmsprop',
             metrics=['accuracy'], # meticts = None 시 loss만 로그 출력
             )
begin = time()
model.fit([X_enc,X_dec],Y_dec,
         epochs=MY_EPOCH,
          workers=-1,
         )
end = time()
print('학습시간 :', end-begin)

Epoch 1/500
4/4 [==============================] - 5s 15ms/step - loss: 5.1107 - accuracy: 0.2242
Epoch 2/500
4/4 [==============================] - 0s 12ms/step - loss: 4.9408 - accuracy: 0.3333
Epoch 3/500
4/4 [==============================] - 0s 12ms/step - loss: 4.2026 - accuracy: 0.3333
Epoch 4/500
4/4 [==============================] - 0s 11ms/step - loss: 3.4683 - accuracy: 0.3333
Epoch 5/500
4/4 [==============================] - 0s 11ms/step - loss: 3.4078 - accuracy: 0.3333
Epoch 6/500
4/4 [==============================] - 0s 12ms/step - loss: 3.3631 - accuracy: 0.3333
Epoch 7/500
4/4 [==============================] - 0s 12ms/step - loss: 3.3286 - accuracy: 0.3333
Epoch 8/500
4/4 [==============================] - 0s 11ms/step - loss: 3.2940 - accuracy: 0.3333
Epoch 9/500
4/4 [==============================] - 0s 9ms/step - loss: 3.2646 - accuracy: 0.3333
Epoch 10/500
4/4 [==============================] - 0s 11ms/step - loss: 3.2464 - accuracy: 0.3333
Epoch 11/500
4/4 [==

4/4 [==============================] - 0s 11ms/step - loss: 0.5040 - accuracy: 0.9576
Epoch 84/500
4/4 [==============================] - 0s 9ms/step - loss: 0.4798 - accuracy: 0.9545
Epoch 85/500
4/4 [==============================] - 0s 10ms/step - loss: 0.4605 - accuracy: 0.9606
Epoch 86/500
4/4 [==============================] - 0s 9ms/step - loss: 0.4444 - accuracy: 0.9667
Epoch 87/500
4/4 [==============================] - 0s 10ms/step - loss: 0.4402 - accuracy: 0.9758
Epoch 88/500
4/4 [==============================] - 0s 10ms/step - loss: 0.4032 - accuracy: 0.9667
Epoch 89/500
4/4 [==============================] - 0s 11ms/step - loss: 0.3879 - accuracy: 0.9697
Epoch 90/500
4/4 [==============================] - 0s 11ms/step - loss: 0.3754 - accuracy: 0.9788
Epoch 91/500
4/4 [==============================] - 0s 12ms/step - loss: 0.3565 - accuracy: 0.9848
Epoch 92/500
4/4 [==============================] - 0s 12ms/step - loss: 0.3465 - accuracy: 0.9788
Epoch 93/500
4/4 [=======

4/4 [==============================] - 0s 10ms/step - loss: 0.0111 - accuracy: 1.0000
Epoch 166/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0106 - accuracy: 1.0000
Epoch 167/500
4/4 [==============================] - 0s 9ms/step - loss: 0.0104 - accuracy: 1.0000
Epoch 168/500
4/4 [==============================] - 0s 9ms/step - loss: 0.0082 - accuracy: 1.0000
Epoch 169/500
4/4 [==============================] - 0s 10ms/step - loss: 0.0076 - accuracy: 1.0000
Epoch 170/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0109 - accuracy: 0.9970
Epoch 171/500
4/4 [==============================] - 0s 7ms/step - loss: 0.0066 - accuracy: 1.0000
Epoch 172/500
4/4 [==============================] - 0s 10ms/step - loss: 0.0114 - accuracy: 0.9970
Epoch 173/500
4/4 [==============================] - 0s 10ms/step - loss: 0.0055 - accuracy: 1.0000
Epoch 174/500
4/4 [==============================] - 0s 10ms/step - loss: 0.0051 - accuracy: 1.0000
Epoch 175/500
4/4 [

4/4 [==============================] - 0s 11ms/step - loss: 8.0741e-05 - accuracy: 1.0000
Epoch 246/500
4/4 [==============================] - 0s 13ms/step - loss: 7.7679e-05 - accuracy: 1.0000
Epoch 247/500
4/4 [==============================] - 0s 8ms/step - loss: 9.1025e-05 - accuracy: 1.0000
Epoch 248/500
4/4 [==============================] - 0s 13ms/step - loss: 1.4026e-04 - accuracy: 1.0000
Epoch 249/500
4/4 [==============================] - 0s 10ms/step - loss: 0.0012 - accuracy: 1.0000
Epoch 250/500
4/4 [==============================] - 0s 10ms/step - loss: 3.5297e-04 - accuracy: 1.0000
Epoch 251/500
4/4 [==============================] - 0s 15ms/step - loss: 5.9860e-05 - accuracy: 1.0000
Epoch 252/500
4/4 [==============================] - 0s 10ms/step - loss: 5.5991e-05 - accuracy: 1.0000
Epoch 253/500
4/4 [==============================] - 0s 8ms/step - loss: 5.2874e-05 - accuracy: 1.0000
Epoch 254/500
4/4 [==============================] - 0s 10ms/step - loss: 4.9991e-05

4/4 [==============================] - 0s 15ms/step - loss: 2.9423e-06 - accuracy: 1.0000
Epoch 325/500
4/4 [==============================] - 0s 8ms/step - loss: 2.8989e-06 - accuracy: 1.0000
Epoch 326/500
4/4 [==============================] - 0s 9ms/step - loss: 2.8184e-06 - accuracy: 1.0000
Epoch 327/500
4/4 [==============================] - 0s 10ms/step - loss: 2.7436e-06 - accuracy: 1.0000
Epoch 328/500
4/4 [==============================] - 0s 11ms/step - loss: 2.7057e-06 - accuracy: 1.0000
Epoch 329/500
4/4 [==============================] - 0s 11ms/step - loss: 2.5991e-06 - accuracy: 1.0000
Epoch 330/500
4/4 [==============================] - 0s 10ms/step - loss: 2.5251e-06 - accuracy: 1.0000
Epoch 331/500
4/4 [==============================] - 0s 11ms/step - loss: 2.5168e-06 - accuracy: 1.0000
Epoch 332/500
4/4 [==============================] - 0s 9ms/step - loss: 2.5410e-06 - accuracy: 1.0000
Epoch 333/500
4/4 [==============================] - 0s 11ms/step - loss: 2.3708e

4/4 [==============================] - 0s 10ms/step - loss: 7.2609e-07 - accuracy: 1.0000
Epoch 404/500
4/4 [==============================] - 0s 11ms/step - loss: 7.2429e-07 - accuracy: 1.0000
Epoch 405/500
4/4 [==============================] - 0s 8ms/step - loss: 7.2104e-07 - accuracy: 1.0000
Epoch 406/500
4/4 [==============================] - 0s 9ms/step - loss: 7.0622e-07 - accuracy: 1.0000
Epoch 407/500
4/4 [==============================] - 0s 12ms/step - loss: 6.9755e-07 - accuracy: 1.0000
Epoch 408/500
4/4 [==============================] - 0s 10ms/step - loss: 6.8888e-07 - accuracy: 1.0000
Epoch 409/500
4/4 [==============================] - 0s 9ms/step - loss: 6.8925e-07 - accuracy: 1.0000
Epoch 410/500
4/4 [==============================] - 0s 10ms/step - loss: 6.8274e-07 - accuracy: 1.0000
Epoch 411/500
4/4 [==============================] - 0s 9ms/step - loss: 6.6721e-07 - accuracy: 1.0000
Epoch 412/500
4/4 [==============================] - 0s 10ms/step - loss: 6.6577e-

4/4 [==============================] - 0s 14ms/step - loss: 3.9628e-07 - accuracy: 1.0000
Epoch 483/500
4/4 [==============================] - 0s 11ms/step - loss: 3.9447e-07 - accuracy: 1.0000
Epoch 484/500
4/4 [==============================] - 0s 10ms/step - loss: 3.8653e-07 - accuracy: 1.0000
Epoch 485/500
4/4 [==============================] - 0s 10ms/step - loss: 3.8761e-07 - accuracy: 1.0000
Epoch 486/500
4/4 [==============================] - 0s 10ms/step - loss: 3.8833e-07 - accuracy: 1.0000
Epoch 487/500
4/4 [==============================] - 0s 11ms/step - loss: 3.8400e-07 - accuracy: 1.0000
Epoch 488/500
4/4 [==============================] - 0s 10ms/step - loss: 3.8183e-07 - accuracy: 1.0000
Epoch 489/500
4/4 [==============================] - 0s 7ms/step - loss: 3.8075e-07 - accuracy: 1.0000
Epoch 490/500
4/4 [==============================] - 0s 10ms/step - loss: 3.7677e-07 - accuracy: 1.0000
Epoch 491/500
4/4 [==============================] - 0s 13ms/step - loss: 3.782

In [18]:
model.evaluate([X_enc,X_dec],Y_dec)

4/4 [==============================] - 1s 7ms/step - loss: 3.5293e-07 - accuracy: 1.0000


[3.529316643380298e-07, 1.0]

## 9. 모델 사용
- 쉬운 문제 : 트레인셋에서 사용한 단어를 그대로 사용
- 문자의 순서를 바꿔서도 예측이 가능한가 체크

In [19]:
# 쉬운 문제 cold, cook, down, cost, copy, duty, date...
easy_test = [['cold','PP'],['cook','PP'],['down','PP'],['cost','PP'],['duty','PP'],['date','PP']]
enc_in, dec_in, dec_out = encoding(easy_test)

In [20]:
# 위의 문제 예측하기
pred = model.predict([np.array(enc_in),np.array(dec_in)])

1/1 [==============================] - 1s 898ms/step


In [21]:
for row,word in zip(pred.argmax(axis=-1).tolist(),easy_test) :
    print(word[0],"=>" ,''.join([alpha[c] for c in row if c not in (0,1,2)]))

cold => 감기
cook => 요리
down => 아래
cost => 비용
duty => 의무
date => 날짜


In [22]:
# 어려운 문제
easy_test = [['clod','PP'],['coko','PP'],['donw','PP'],['cots','PP'],['duyt','PP'],['daet','PP'],['love','PP'],['loev','PP'],['lvoe','PP'],
            ['leov','PP'],['eolv','PP'],['evol','PP'],['olve','PP']]
enc_in, dec_in, dec_out = encoding(easy_test)

In [23]:
pred = model.predict([np.array(enc_in),np.array(dec_in)])

1/1 [==============================] - 0s 37ms/step


In [24]:
for row,word in zip(pred.argmax(axis=-1).tolist(),easy_test) :
    print(word[0],"=>" ,''.join([alpha[c] for c in row if c not in (0,1,2)]))

clod => 감기
coko => 요리
donw => 아래
cots => 비용
duyt => 의무
daet => 날짜
love => 사랑
loev => 사랑
lvoe => 사랑
leov => 왼랑
eolv => 도다
evol => 매도
olve => 사랑


In [25]:
# Python
import pandas as pd
from prophet import Prophet

df = pd.read_csv('https://raw.githubusercontent.com/facebook/prophet/main/examples/example_wp_log_peyton_manning.csv')
display(df.tail()) # 날짜에 따른 y값 변동, 2007 ~ 2016

m = Prophet()
m.fit(df)
m # 예측 모델?

future = m.make_future_dataframe(periods=365) # 미래를 그려주는?
future.tail()

# forecast = m.predict(future)
# forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

# fig1 = m.plot(forecast)

Importing plotly failed. Interactive plots will not work.


,ds,y
2900,2016-01-16,7.817223
2901,2016-01-17,9.273878
2902,2016-01-18,10.333775
2903,2016-01-19,9.125871
2904,2016-01-20,8.891374


11:13:30 - cmdstanpy - INFO - Chain [1] start processing
11:13:31 - cmdstanpy - INFO - Chain [1] done processing


,ds
3265,2017-01-15
3266,2017-01-16
3267,2017-01-17
3268,2017-01-18
3269,2017-01-19
